# Notebook 04 — Skill Matching & Gap Scoring
**Capstone Project CC26-PRU469**: Skill Gap Simulator

---

## Tujuan
1. Mendemonstrasikan logika **skill matching** antara profil user dan kebutuhan role
2. Menjelaskan formula **Weighted Gap Score** secara detail
3. Menghitung **estimasi waktu siap kerja** dengan variasi skenario
4. Membandingkan hasil antar skenario pasar (Career Scenario Engine)
5. Menyiapkan data final untuk modeling dan dashboard

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, json, warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
PROCESSED = os.path.join('..', 'data', 'processed')
REPORTS = os.path.join('..', 'reports')

df_skill_master = pd.read_csv(os.path.join(PROCESSED, 'skill_master_cleaned.csv'))
df_users = pd.read_csv(os.path.join(PROCESSED, 'user_profiles_cleaned.csv'))
df_user_skills = pd.read_csv(os.path.join(PROCESSED, 'user_skills_cleaned.csv'))
print('Data loaded.')

## 1. Formula Weighted Gap Score

### Definisi
$$\text{Gap Score} = 1 - \frac{\sum_{i=1}^{n} (\text{user\_skill}_i \times \text{importance}_i)}{\sum_{i=1}^{n} \text{importance}_i}$$

### Estimasi Waktu
$$\text{Weeks} = \frac{\sum_{i=1}^{n} (\text{gap}_i \times \text{hours}_i \times \text{importance}_i)}{\text{study\_hours\_per\_week}} \times \text{market\_multiplier}$$

| Skenario | Multiplier |
|---|---|
| Normal | 1.0x |
| Kompetitif | 1.3x |
| Booming | 0.8x |

## 2. Demonstrasi: Simulasi untuk Satu User

In [ ]:
# Ambil contoh user
sample_user = df_users.iloc[0]
print(f'User ID       : {sample_user["user_id"]}')
print(f'Target Role   : {sample_user["target_role"]}')
print(f'Background    : {sample_user["background_level"]}')
print(f'Study Hours   : {sample_user["study_hours_per_week"]} jam/minggu')
print(f'Scenario      : {sample_user["market_scenario"]}')
print(f'Gap Score     : {sample_user["gap_score"]}')
print(f'Readiness     : {sample_user["readiness_label"]}')
print(f'Top Missing   : {sample_user["top_missing_skill_1"]}, {sample_user["top_missing_skill_2"]}, {sample_user["top_missing_skill_3"]}')
print(f'Est. Weeks    : {sample_user["estimated_weeks_ready"]}')

In [ ]:
# Visualisasi profil skill user vs kebutuhan role
role = sample_user['target_role']
user_skills_data = df_user_skills[df_user_skills['user_id'] == sample_user['user_id']]
role_requirements = df_skill_master[df_skill_master['role'] == role]

# Merge
comparison = role_requirements.merge(
    user_skills_data[['skill_name', 'proficiency_level']], 
    on='skill_name', how='left'
).fillna(0)

comparison['required_level'] = comparison['importance_score'] * 100  # scale to 0-100
comparison['gap'] = comparison['required_level'] - comparison['proficiency_level']
comparison['gap'] = comparison['gap'].clip(lower=0)

# Plot
fig, ax = plt.subplots(figsize=(14, 8))
comp_sorted = comparison.sort_values('importance_score', ascending=True)

y_pos = range(len(comp_sorted))
ax.barh(y_pos, comp_sorted['required_level'], alpha=0.3, color='red', label='Required Level', height=0.7)
ax.barh(y_pos, comp_sorted['proficiency_level'], alpha=0.7, color='#2196F3', label='User Proficiency', height=0.5)

ax.set_yticks(y_pos)
ax.set_yticklabels(comp_sorted['skill_name'])
ax.set_xlabel('Level (0-100)')
ax.set_title(f'Skill Gap Analysis — {sample_user["user_id"]} (Target: {role})', fontweight='bold')
ax.legend(loc='lower right')

plt.tight_layout()
plt.savefig(os.path.join(REPORTS, 'gap_01_individual_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

## 3. Career Scenario Engine — Perbandingan 3 Skenario

In [ ]:
def compute_gap_score(user_proficiency, skill_master, role, scenario='Normal'):
    """Hitung gap score untuk skenario tertentu."""
    multiplier = {'Normal': 1.0, 'Kompetitif': 1.15, 'Booming': 0.90}
    m = multiplier[scenario]
    
    skills = skill_master[skill_master['role'] == role]
    weighted_sum = 0
    importance_sum = 0
    
    for _, row in skills.iterrows():
        imp = min(1.0, row['importance_score'] * m)
        user_level = user_proficiency.get(row['skill_name'], 0) / 100.0
        weighted_sum += user_level * imp
        importance_sum += imp
    
    return round(1 - (weighted_sum / importance_sum), 4) if importance_sum > 0 else 1.0

def estimate_weeks(user_proficiency, skill_master, role, study_hours, scenario='Normal'):
    """Estimasi minggu siap kerja."""
    multiplier = {'Normal': 1.0, 'Kompetitif': 1.3, 'Booming': 0.8}
    m = multiplier[scenario]
    
    skills = skill_master[skill_master['role'] == role]
    total_hours = 0
    
    for _, row in skills.iterrows():
        user_level = user_proficiency.get(row['skill_name'], 0) / 100.0
        gap = max(0, 1 - user_level)
        total_hours += gap * row['avg_learning_hours'] * row['importance_score']
    
    return round((total_hours / study_hours) * m, 1) if study_hours > 0 else 999

# Demo: Simulasi 3 skenario untuk sample user
user_skills_json = json.loads(sample_user['current_skills_json'])
study_h = sample_user['study_hours_per_week']

print(f'\nSimulasi untuk {sample_user["user_id"]} (Target: {role})')
print(f'Study hours: {study_h} jam/minggu\n')

scenarios = ['Normal', 'Kompetitif', 'Booming']
results = []
for sc in scenarios:
    gap = compute_gap_score(user_skills_json, df_skill_master, role, sc)
    weeks = estimate_weeks(user_skills_json, df_skill_master, role, study_h, sc)
    results.append({'Skenario': sc, 'Gap Score': gap, 'Estimasi Minggu': weeks})
    print(f'  {sc:12s} | Gap: {gap:.4f} | Weeks: {weeks}')

df_results = pd.DataFrame(results)
df_results

In [ ]:
# Visualisasi perbandingan skenario
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sc_colors = ['#2196F3', '#FF5722', '#4CAF50']

axes[0].bar(df_results['Skenario'], df_results['Gap Score'], color=sc_colors)
axes[0].set_title('Gap Score per Skenario', fontweight='bold')
axes[0].set_ylabel('Gap Score')
axes[0].set_ylim(0, 1)

axes[1].bar(df_results['Skenario'], df_results['Estimasi Minggu'], color=sc_colors)
axes[1].set_title('Estimasi Minggu per Skenario', fontweight='bold')
axes[1].set_ylabel('Minggu')

plt.suptitle(f'Career Scenario Engine — {sample_user["user_id"]}',
             fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(REPORTS, 'gap_02_scenario_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4. Aggregate Analysis — Gap Score per Role & Scenario

In [ ]:
# Tabel pivot summary
summary = df_users.pivot_table(
    values=['gap_score', 'estimated_weeks_ready'],
    index=['target_role', 'background_level'],
    columns='market_scenario',
    aggfunc='mean'
).round(2)

print('=== Summary: Rata-rata Gap Score & Estimasi Minggu ===')
summary

In [ ]:
# Heatmap: Gap Score per Role x Background x Scenario
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
bg_order = ['Pemula', 'Menengah', 'Lanjutan']

for i, sc in enumerate(scenarios):
    subset = df_users[df_users['market_scenario'] == sc]
    pivot = subset.pivot_table(values='gap_score', index='background_level',
                               columns='target_role', aggfunc='mean')
    pivot = pivot.reindex(bg_order)
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn_r', ax=axes[i],
                vmin=0, vmax=1, linewidths=1)
    axes[i].set_title(f'Skenario: {sc}', fontweight='bold')
    axes[i].set_ylabel('' if i > 0 else 'Background Level')

plt.suptitle('Gap Score Heatmap per Skenario Pasar', fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(REPORTS, 'gap_03_heatmap_scenarios.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. Export Data Final untuk Modeling & Dashboard

In [ ]:
# Dataset final yang siap dipakai oleh AI Engineer untuk modeling
# dan oleh dashboard Streamlit

# 5.1 Modeling dataset: user features + target
user_features = df_user_skills.pivot_table(
    index='user_id', columns='skill_name', values='proficiency_level'
).reset_index()

# Merge dengan user metadata
df_modeling = df_users[['user_id', 'target_role', 'background_level', 
                         'study_hours_per_week', 'market_scenario',
                         'gap_score', 'estimated_weeks_ready', 'readiness_label']].merge(
    user_features, on='user_id', how='left'
)

# Fill NaN skill columns with 0 (user tidak punya skill tersebut)
df_modeling = df_modeling.fillna(0)

# Save
df_modeling.to_csv(os.path.join(PROCESSED, 'modeling_dataset.csv'), index=False)
print(f'Modeling dataset: {df_modeling.shape}')
print(f'Saved to: data/processed/modeling_dataset.csv')

# 5.2 Verify
print(f'\nKolom: {list(df_modeling.columns[:10])} ... ({len(df_modeling.columns)} total)')
df_modeling.head(3)

In [ ]:
print('\n=== SEMUA NOTEBOOK DATA SCIENCE SELESAI ===')
print(f'\nFile di data/processed/:')
for f in sorted(os.listdir(PROCESSED)):
    size = os.path.getsize(os.path.join(PROCESSED, f))
    print(f'  {f:40s} {size:>10,} bytes')

print(f'\nFile di reports/:')
for f in sorted(os.listdir(REPORTS)):
    size = os.path.getsize(os.path.join(REPORTS, f))
    print(f'  {f:40s} {size:>10,} bytes')